In [0]:

from urllib.request import urlretrieve
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
%pip install openpyxl

In [0]:

# Databricks notebook source

from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import hashlib
import time
import zipfile

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType,
    DoubleType,
)

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

BASE = Path("/Volumes/workspace/bronze/raw")

TABELA = "workspace.observability.raw_ingestion_runs"

arquivos = {
    "ibge": {
        "agregados_basicos.zip":
            "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Distrito_csv/Agregados_por_distritos_basico_BR_20260520.zip",
    },
    "ssp": {
        "SPDadosCriminais_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/SPDadosCriminais_2026.xlsx",

        "DadosProdutividade_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/DadosProdutividade_2026.xlsx",

        "MDIP_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/MDIP_2026.xlsx",

        "CelularesSubtraidos_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/celularesSub/CelularesSubtraidos_2026.xlsx",

        "VeiculosSubtraidos_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/veiculosSub/VeiculosSubtraidos_2026.xlsx",

        "ObjetosSubtraidos_2026.xlsx":
            "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/objetosSub/ObjetosSubtraidos_2026.xlsx",
    },
    "observasampa": {
        "ObservaSampaDadosAbertosIndicadoresCSV.csv":
            "https://dados-abertos-observasampa.prefeitura.sp.gov.br/_temp/DadosAbertos/ObservaSampaDadosAbertosIndicadoresCSV.csv",
    },
    "imoveis": {
        "gurudosimoveis.csv":
            "https://www.gurudosimoveis.com.br/dados/sao-paulo-sp.csv",
    },
}

FONTES_ESPERADAS = {
    "ibge",
    "ssp",
    "observasampa",
    "imoveis",
}

assert str(BASE) == "/Volumes/workspace/bronze/raw"
assert set(arquivos.keys()) == FONTES_ESPERADAS


# ============================================================
# 2. TABELA DELTA DE AUDITORIA
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA} (
    run_id STRING,
    executado_em TIMESTAMP,
    fonte STRING,
    arquivo STRING,
    url STRING,
    status STRING,
    tamanho_bytes BIGINT,
    sha256 STRING,
    etag STRING,
    last_modified STRING,
    duracao_segundos DOUBLE,
    erro STRING
)
USING DELTA
""")

schema_auditoria = StructType([
    StructField("run_id", StringType(), True),
    StructField("executado_em", TimestampType(), True),
    StructField("fonte", StringType(), True),
    StructField("arquivo", StringType(), True),
    StructField("url", StringType(), True),
    StructField("status", StringType(), True),
    StructField("tamanho_bytes", LongType(), True),
    StructField("sha256", StringType(), True),
    StructField("etag", StringType(), True),
    StructField("last_modified", StringType(), True),
    StructField("duracao_segundos", DoubleType(), True),
    StructField("erro", StringType(), True),
])


def registrar_auditoria(registro):
    df = spark.createDataFrame(
        [registro],
        schema=schema_auditoria
    )

    df.write.mode("append").saveAsTable(TABELA)


def ultimo_registro(fonte, nome):
    return (
        spark.table(TABELA)
        .filter(
            (F.col("fonte") == fonte)
            & (F.col("arquivo") == nome)
            & F.col("status").isin(
                "NOVO",
                "ATUALIZADO",
                "SEM_ALTERACAO",
                "NAO_MODIFICADO",
            )
        )
        .orderBy(F.col("executado_em").desc())
        .first()
    )


# ============================================================
# 3. SESSÃO HTTP COM RETENTATIVAS
# ============================================================

sessao = requests.Session()

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=["GET"],
)

sessao.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

sessao.mount(
    "http://",
    HTTPAdapter(max_retries=retry)
)


# ============================================================
# 4. FUNÇÕES DE INTEGRIDADE
# ============================================================

def calcular_hash(caminho):
    """Calcula SHA-256 sem carregar todo o arquivo na memória."""

    h = hashlib.sha256()

    with caminho.open("rb") as f:
        for bloco in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloco)

    return h.hexdigest()


def validar_arquivo(caminho, nome):
    """Verifica integridade básica do arquivo baixado."""

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    if caminho.stat().st_size == 0:
        raise ValueError(
            f"Arquivo vazio: {nome}"
        )

    extensao = Path(nome).suffix.lower()

    if extensao in (".zip", ".xlsx"):

        if not zipfile.is_zipfile(caminho):
            raise ValueError(
                f"Arquivo ZIP/XLSX inválido: {nome}"
            )

        with zipfile.ZipFile(caminho) as z:
            corrompido = z.testzip()

            if corrompido:
                raise ValueError(
                    f"Conteúdo corrompido: {corrompido}"
                )

    elif extensao == ".csv":

        with caminho.open("rb") as f:
            inicio = f.read(1024).strip().lower()

        if (
            b"<html" in inicio
            or b"<!doctype html" in inicio
        ):
            raise ValueError(
                f"Resposta HTML no lugar de CSV: {nome}"
            )


# ============================================================
# 5. CORRIGIR NOMES LEGADOS DA SSP
# ============================================================

# O notebook anterior salvava dois downloads de 2026
# com nomes terminados em 2022.
# Renomeamos apenas se o arquivo novo ainda não existir.

renomeacoes = {
    "VeiculosSubtraidos_2022.xlsx":
        "VeiculosSubtraidos_2026.xlsx",

    "ObjetosSubtraidos_2022.xlsx":
        "ObjetosSubtraidos_2026.xlsx",
}

pasta_ssp = BASE / "ssp"

if pasta_ssp.exists():

    for antigo, novo in renomeacoes.items():

        origem = pasta_ssp / antigo
        destino = pasta_ssp / novo

        if origem.exists() and not destino.exists():

            validar_arquivo(origem, antigo)

            origem.rename(destino)

            print(
                f"[RENOMEADO] {antigo} -> {novo}"
            )

        elif origem.exists() and destino.exists():

            print(
                f"[AVISO] Ambos existem: {antigo} e {novo}. "
                "Nenhum foi excluído."
            )


# ============================================================
# 6. INGESTÃO INCREMENTAL
# ============================================================

falhas = []
resultados = []

for fonte, bases in arquivos.items():

    pasta = BASE / fonte
    pasta.mkdir(parents=True, exist_ok=True)

    for nome, url in bases.items():

        inicio = time.perf_counter()
        run_id = str(uuid4())

        destino = pasta / nome

        temporario = (
            pasta / f".{nome}.{run_id}.part"
        )

        status = "ERRO"
        erro = None
        tamanho = None
        hash_final = None
        etag = None
        last_modified = None

        try:

            # ----------------------------------------
            # 6.1. Histórico anterior
            # ----------------------------------------

            anterior = ultimo_registro(
                fonte,
                nome
            )

            headers = {
                "User-Agent": "Mozilla/5.0"
            }

            # ----------------------------------------
            # 6.2. Verificar integridade local
            # ----------------------------------------

            arquivo_integro = False

            if anterior and destino.exists():

                hash_esperado = anterior["sha256"]

                if hash_esperado:

                    hash_local = calcular_hash(
                        destino
                    )

                    arquivo_integro = (
                        hash_local == hash_esperado
                    )

                if not arquivo_integro:
                    print(
                        f"[AVISO] Hash local divergente "
                        f"ou desconhecido: {nome}"
                    )

            # ----------------------------------------
            # 6.3. HTTP condicional
            # ----------------------------------------

            if anterior and arquivo_integro:

                if anterior["etag"]:

                    headers["If-None-Match"] = (
                        anterior["etag"]
                    )

                if anterior["last_modified"]:

                    headers["If-Modified-Since"] = (
                        anterior["last_modified"]
                    )

            # ----------------------------------------
            # 6.4. Download
            # ----------------------------------------

            with sessao.get(
                url,
                headers=headers,
                timeout=(20, 120),
                stream=True,
            ) as resposta:

                # ------------------------------------
                # HTTP 304: fonte não modificada
                # ------------------------------------

                if resposta.status_code == 304:

                    if not anterior or not destino.exists():

                        raise FileNotFoundError(
                            "HTTP 304 sem arquivo local"
                        )

                    hash_local = calcular_hash(
                        destino
                    )

                    if (
                        not anterior["sha256"]
                        or hash_local != anterior["sha256"]
                    ):

                        raise ValueError(
                            "Arquivo local divergente "
                            "após HTTP 304"
                        )

                    status = "NAO_MODIFICADO"

                    tamanho = destino.stat().st_size

                    hash_final = hash_local

                # ------------------------------------
                # HTTP 200: conteúdo disponível
                # ------------------------------------

                else:

                    resposta.raise_for_status()

                    h = hashlib.sha256()
                    total = 0

                    with temporario.open("wb") as f:

                        for bloco in resposta.iter_content(
                            chunk_size=1024 * 1024
                        ):

                            if not bloco:
                                continue

                            f.write(bloco)
                            h.update(bloco)

                            total += len(bloco)

                    # --------------------------------
                    # Verificar tamanho recebido
                    # --------------------------------

                    comprimento = (
                        resposta.headers.get(
                            "Content-Length"
                        )
                    )

                    codificacao = (
                        resposta.headers.get(
                            "Content-Encoding",
                            "identity"
                        ).lower()
                    )

                    if (
                        comprimento
                        and codificacao == "identity"
                    ):

                        if total != int(comprimento):

                            raise ValueError(
                                "Tamanho recebido diferente "
                                "do Content-Length"
                            )

                    # --------------------------------
                    # Validar novo arquivo
                    # --------------------------------

                    validar_arquivo(
                        temporario,
                        nome
                    )

                    novo_hash = h.hexdigest()

                    # --------------------------------
                    # Comparar com versão existente
                    # --------------------------------

                    if destino.exists():

                        hash_antigo = calcular_hash(
                            destino
                        )

                        if hash_antigo == novo_hash:

                            status = "SEM_ALTERACAO"

                            temporario.unlink()

                        else:

                            # Só substitui depois de
                            # validar o novo conteúdo.
                            temporario.replace(
                                destino
                            )

                            status = "ATUALIZADO"

                    else:

                        temporario.replace(
                            destino
                        )

                        status = "NOVO"

                    tamanho = (
                        destino.stat().st_size
                    )

                    hash_final = novo_hash

                # ------------------------------------
                # 6.5. Metadados HTTP
                # ------------------------------------

                etag = resposta.headers.get(
                    "ETag"
                )

                last_modified = (
                    resposta.headers.get(
                        "Last-Modified"
                    )
                )

                # No HTTP 304, preservar metadados
                # anteriores quando não retornados.

                if (
                    resposta.status_code == 304
                    and anterior
                ):

                    etag = (
                        etag
                        or anterior["etag"]
                    )

                    last_modified = (
                        last_modified
                        or anterior["last_modified"]
                    )

        # --------------------------------------------
        # 6.6. Tratamento de erro
        # --------------------------------------------

        except Exception as e:

            erro = str(e)

            status = "ERRO"

            falhas.append(
                f"{fonte}/{nome}: {erro}"
            )

        # --------------------------------------------
        # 6.7. Limpeza e auditoria
        # --------------------------------------------

        finally:

            if temporario.exists():
                temporario.unlink()

            duracao = round(
                time.perf_counter() - inicio,
                2
            )

            registro = (
                run_id,
                datetime.now(timezone.utc),
                fonte,
                nome,
                url,
                status,
                tamanho,
                hash_final,
                etag,
                last_modified,
                duracao,
                erro,
            )

            registrar_auditoria(
                registro
            )

            resultados.append({
                "fonte": fonte,
                "arquivo": nome,
                "status": status,
                "tamanho_bytes": tamanho,
                "duracao_segundos": duracao,
                "erro": erro,
            })

            print(
                f"[{status}] "
                f"{fonte}/{nome} "
                f"({duracao}s)"
            )

            if erro:
                print(
                    f"  ERRO: {erro}"
                )


# ============================================================
# 7. RESUMO DA EXECUÇÃO
# ============================================================

print("\n" + "=" * 60)
print("BairroIQ - Resumo da Ingestão Bronze")
print("=" * 60)

for status_possivel in [
    "NOVO",
    "ATUALIZADO",
    "SEM_ALTERACAO",
    "NAO_MODIFICADO",
    "ERRO",
]:

    quantidade = sum(
        1
        for r in resultados
        if r["status"] == status_possivel
    )

    print(
        f"{status_possivel}: {quantidade}"
    )

print(f"TOTAL: {len(resultados)}")
print(f"FALHAS: {len(falhas)}")


# ============================================================
# 8. CONSULTAR AUDITORIA
# ============================================================

display(
    spark.table(TABELA)
    .orderBy(
        F.col("executado_em").desc()
    )
    .limit(100)
)


# ============================================================
# 9. RESULTADO FINAL
# ============================================================

sessao.close()

if falhas:

    raise RuntimeError(
        "Ingestão Bronze concluída com falhas:\n"
        + "\n".join(falhas)
    )

print(
    "\nIngestão Bronze finalizada com sucesso!"
)
